# MTP-1 -> MTP-2 Live Pipeline

Runs the whole project **live, cell by cell** -- no GPU, no pre-existing results trusted on faith. Every table below is computed on the fly from real files in this repo. Runs **locally** (VS Code / Jupyter) **or in Google Colab** -- see the Setup cells below.

**Scope split, exactly as scoped with the supervisor:**
- **MTP-1 = Parts 1-4 only.** This is the replication of Ranjan & van Schijndel (2024): load the treebank, filter it, generate word-order variants, extract the paper's original 5 features, and reproduce its reported accuracy with the same 5-feature/LSTM pairwise classifier.
- **MTP-2 = Parts 5-8.** Everything beyond the replication -- swapping the LSTM surprisal feature for large-language-model surprisal (XGLM-1.7B, Gemma-3), and adding 3 new syntactic-surprisal features -- is MTP-2 work, not MTP-1.

**The only hardcoded numbers in this entire notebook** are Ranjan & van Schijndel (2024)'s own *published* paper results -- their dataset composition table (Part 2, Table 1c) and their reported accuracy (Part 4, `PAPER_ACC`) -- used purely as external comparison baselines, flagged explicitly where they appear. Everything else -- every accuracy, every coefficient, every p-value -- is fit fresh by scikit-learn / statsmodels when you run the cell.

1. **Part 1** -- `[MTP-1]` load the raw UD_Hindi-HDTB treebank, filter it, generate word-order variants (live correctness check against the cached dataset)
2. **Part 2** -- `[MTP-1]` load the authoritative merged feature table (`features.csv`) and the corpus's order-type distribution, presented the way the paper presents it
3. **Part 3** -- `[MTP-1]` shared pairwise-classifier helpers
4. **Part 4** -- `[MTP-1]` replication check vs. the published paper (Table 2) -- **this is the end of MTP-1**
5. **Part 5** -- `[MTP-2]` full 15-experiment matrix: LSTM vs. XGLM vs. Gemma-3, plus the 3 new tool features (PCFG/Modelblocks/Dep-parser), alone and combined (Table 3 + Figure 1)
6. **Part 6** -- `[MTP-2]` GLM coefficient significance for every model (Table 4)
7. **Part 7** -- `[MTP-2]` why LLMs win: stratified by dependency length AND out-of-vocabulary rate (Table 5 + Figure 2)
8. **Part 8** -- final summary across MTP-1 and MTP-2, side by side (Figure 3)

Run every cell top to bottom (Kernel -> Restart & Run All). Total runtime is well under a minute locally -- nothing here needs a GPU; the expensive LLM/parsing steps were already done once and are cached in this repo's own `models/` and `results/` folders.

## Setup

**Running locally:** no action needed -- the defaults below find the checkout automatically.

**Running in Google Colab:** upload (or sync via `My Drive`) **just this one folder** --
`hindi_wordorder_replication` -- to your Drive root. You don't need the multi-GB
trained-model files either -- only `src/`, `data/ud_hindi/train.conllu`,
`data/processed/features/features.csv`, and `models/adaptive_lstm/vocab.json` are actually
read by this notebook. The first code cell below detects Colab automatically, mounts your Drive, and
installs any missing packages; edit `PROJECT_ROOT` in the second cell only if you uploaded it
somewhere other than your Drive root.

In [1]:
import os

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q',
                    'pandas', 'numpy', 'scipy', 'scikit-learn', 'statsmodels', 'matplotlib'])

print('Running in Colab:', IN_COLAB)

Running in Colab: False


In [2]:
# This notebook expects to run from its own `notebooks/` folder inside the
# hindi_wordorder_replication checkout (local case), or with that one folder
# uploaded to your Drive root (Colab case).
if IN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/hindi_wordorder_replication'  # edit if uploaded elsewhere
else:
    cwd = os.getcwd()
    PROJECT_ROOT = os.path.dirname(cwd) if os.path.basename(cwd).lower() == 'notebooks' else cwd
    if not os.path.isdir(os.path.join(PROJECT_ROOT, 'src')):
        PROJECT_ROOT = r'E:\hindi_wordorder_replication'  # edit this line if your checkout lives elsewhere

print('PROJECT_ROOT:', PROJECT_ROOT)
assert os.path.isdir(os.path.join(PROJECT_ROOT, 'src')), (
    f'src/ not found under {PROJECT_ROOT} -- fix PROJECT_ROOT above '
    '(in Colab: make sure the folder is uploaded/synced to your Drive at this path)')

In [3]:
TRAIN_CONLLU_DIR  = os.path.join(PROJECT_ROOT, 'data', 'ud_hindi')
FEATURES_CSV      = os.path.join(PROJECT_ROOT, 'data', 'processed', 'features', 'features.csv')
VOCAB_JSON        = os.path.join(PROJECT_ROOT, 'models', 'adaptive_lstm', 'vocab.json')
FIGURES_DIR       = os.path.join(PROJECT_ROOT, 'results', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

required = {
    'train.conllu'        : os.path.join(TRAIN_CONLLU_DIR, 'train.conllu'),
    'features.csv'        : FEATURES_CSV,
    'adaptive_lstm vocab' : VOCAB_JSON,
}
missing = {name: path for name, path in required.items() if not os.path.exists(path)}
if missing:
    raise FileNotFoundError(f'Missing required files -- edit the paths above to match where they actually are: {missing}')
print('All required files found:')
for name, path in required.items():
    print(f'  {name:<24} -> {path}')

In [4]:
import sys, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from scipy import stats as scipy_stats
%matplotlib inline

sys.path.insert(0, PROJECT_ROOT)
from src.data_loading.loader import HUTBLoader
from src.preprocessing.filter import SentenceFilter
from src.variant_generation.generator import VariantGenerator
from src.variant_generation.grammar_filter import GrammarFilter
from src.ranking.ranking import build_pairwise_dataset
from src.ranking.classifier import PairwiseClassifier

# Validated 3-series categorical palette, fixed order (blue / orange / aqua) + purple for new features
C_LSTM, C_XGLM, C_GEMMA, C_NEW = '#2a78d6', '#eb6834', '#1baf7a', '#9b59b6'
INK, INK_MUTED, GRIDLINE = '#0b0b0b', '#898781', '#e1e0d9'

def style_axes(ax):
    for s in ('top', 'right', 'left'):
        ax.spines[s].set_visible(False)
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(left=False)

def bar_labels(ax, bars, fmt='{:.1f}'):
    for b in bars:
        h = b.get_height()
        ax.annotate(fmt.format(h), (b.get_x() + b.get_width() / 2, h),
                    textcoords='offset points', xytext=(0, 3),
                    ha='center', va='bottom', fontsize=8.5, color=INK)

print('Ready.')

Ready.


## Part 1 [MTP-1] -- Load + filter + generate, live (correctness check)

Reruns the real pipeline from raw text: load the treebank, apply the paper's sentence filter, generate grammatical word-order variants. This should reproduce the same totals that `features.csv` was originally built from -- a live correctness check, not a formality.

In [5]:
loader = HUTBLoader(TRAIN_CONLLU_DIR)
raw = loader.load_train()
sentences = [s['tokens'] for s in raw]
print(f'Raw sentences loaded from train.conllu: {len(raw):,}')

Loading: E:\hindi_wordorder_replication\data\ud_hindi\train.conllu


Loaded 13306 sentences from train.conllu
Raw sentences loaded from train.conllu: 13,306


In [6]:
filt = SentenceFilter()
filtered, rejected_df = filt.filter_with_rejection_tracking(sentences)

print(f'Passed filter: {len(filtered):,}')
print('Rejection breakdown:')
for reason, count in filt.stats['rejected'].items():
    if count > 0:
        print(f'  {reason:<22}: {count}')

Passed filter: 2,320
Rejection breakdown:
  non_declarative       : 66
  negation              : 1705
  non_projective        : 1451
  no_verb_root          : 1290
  no_subj_obj           : 5019
  few_preverbal         : 912
  so_not_preverbal      : 543


In [7]:
import io, contextlib

gf = GrammarFilter()
gf.build_from_corpus(filtered)
print(f'Attested transition pairs: {len(gf.attested)}')

metadata_df = pd.DataFrame([
    {'sent_id': raw[i].get('sent_id') or f'sent_{i:04d}'}
    for i in range(len(filtered))
])

gen = VariantGenerator(max_variants=99, grammar_filter=gf)
_buf = io.StringIO()
with contextlib.redirect_stdout(_buf):
    live_variants_df = gen.generate_for_all(filtered, metadata_df, output_dir=None)

_captured = _buf.getvalue()
_cutoff = _captured.find('Paper target (total)')
_end = _captured.find('\n', _cutoff) + 1 if _cutoff != -1 else len(_captured)
print(_captured[:_end], end='')

live_refs = live_variants_df[live_variants_df['is_original'] == 1]
live_vars = live_variants_df[live_variants_df['is_original'] == 0]

## Part 2 [MTP-1] -- Load the authoritative merged feature table

`data/processed/features/features.csv` is the single merged dataset `scripts/assemble_features.py` already built: dependency length and IS-score come straight from Part 1's generation step, and every surprisal column (trigram, LSTM, XGLM-1.7B, Gemma-3, PCFG-tool, Modelblocks, Dep-parser) was merged in from each model's own cached scoring run. Nothing is re-merged here -- this cell just loads the result.

In [8]:
df = pd.read_csv(FEATURES_CSV)
print(f'Loaded {len(df):,} rows, {len(df.columns)} columns:')
print(list(df.columns))

Loaded 78,865 rows, 16 columns:
['reference_id', 'variant_id', 'is_original', 'is_flip_pair', 'deprel_sequence', 'order_type', 'dl', 'text', 'surprisal_trigram', 'is_score', 'surprisal_lstm', 'surprisal_xglm', 'surprisal_gemma', 'surprisal_pcfg', 'surprisal_modelblocks', 'surprisal_depparse']


### Table 1a -- Order-type distribution (as presented in the paper)

Reference sentences and their paired variants, broken down by construction type -- same Pattern / Reference / Variant layout as Tables 1b and 1c, computed live from our own data.

In [9]:
PATTERNS_1A = {
    'DO-fronted': 'Ref: DOSV; Var: SDOV',
    'IO-fronted': 'Ref: IOSV; Var: SIOV',
    'Canonical':  'Ref: SOV; Var: OSV',
}
ROW_ORDER_1A = ['DO-fronted', 'IO-fronted', 'Canonical']  # matches Tables 1b/1c's row order

refs_1a = df[df['is_original'] == 1].groupby('order_type')['reference_id'].nunique().reindex(ROW_ORDER_1A)
n_refs_1a = int(refs_1a.sum())

def variant_count_1a(order_type):
    ref_ids = df[(df['is_original'] == 1) & (df['order_type'] == order_type)]['reference_id'].unique()
    sub = df[df['reference_id'].isin(ref_ids)]
    if order_type == 'Canonical':
        return len(sub[(sub['is_original'] == 0) & (sub['order_type'] != 'Canonical')])
    return len(sub[(sub['is_original'] == 0) & (sub['order_type'] == 'Canonical')])

variant_counts_1a = pd.Series({o: variant_count_1a(o) for o in ROW_ORDER_1A})
n_vars_1a = int(variant_counts_1a.sum())
n_vars_total_1a = int((df['is_original'] == 0).sum())

table1a = pd.DataFrame({
    'Pattern':   pd.Series(PATTERNS_1A).reindex(ROW_ORDER_1A),
    'Reference': [f'{int(refs_1a[o]):,} ({refs_1a[o] / n_refs_1a * 100:.2f}%)' for o in ROW_ORDER_1A],
    'Variant':   [f'{int(variant_counts_1a[o]):,} ({variant_counts_1a[o] / n_vars_1a * 100:.2f}%)' for o in ROW_ORDER_1A],
}, index=ROW_ORDER_1A)
table1a.loc['Full dataset'] = ['', f'{n_refs_1a:,}', f'{n_vars_total_1a:,}']
table1a.index.name = 'Dataset ordering type'

print('Order distribution:')
for order in ROW_ORDER_1A:
    print(f'  {order:<14}: {table1a.loc[order, "Reference"]:>16}    {table1a.loc[order, "Variant"]:>16}')
table1a

### Table 1b -- reference sentences / variants per construction type

Same column layout as Table 1c (Pattern / Reference / Variant, same row order: DO-fronted,
IO-fronted, Canonical) for direct side-by-side comparison -- computed live from our own data,
**using the paper's own definition of "Variant"**: not every variant labelled that type, but the
specific N actually paired against that construction's references in the pairwise classifier
(DO-fronted/IO-fronted rows: count of SDOV/SIOV *canonical* variants generated from those
references -- matching `src/ranking/ranking.py`'s own "DOSV corpus vs SDOV canonical variants,
N~1663" / "IOSV corpus vs SIOV canonical variants, N~1353" documentation; Canonical row: count of
non-canonical, fronted-type variants generated from canonical references, the reverse-direction
comparison). This is why the paper's own Variant column doesn't sum to its stated "Full dataset"
total either (1,663+1,353+30,851 = 33,867, not 72,833) -- it's a per-comparison N, not a
partition of all variants.

Our DO-fronted/IO-fronted counts (1,703 / 1,292) land very close to the paper's (1,663 / 1,353),
confirming the methodology lines up; the Canonical row and the totals differ more because our
corpus has 2,320 references vs. the paper's 1,996 and no separate `O-fronted` category (retired
-- see the README's "A note on data history").

In [10]:
PATTERNS = {
    'DO-fronted': 'Ref: DOSV; Var: SDOV',
    'IO-fronted': 'Ref: IOSV; Var: SIOV',
    'Canonical':  'Ref: SOV; Var: OSV',
}
ROW_ORDER = ['DO-fronted', 'IO-fronted', 'Canonical']

refs = df[df['is_original'] == 1].groupby('order_type')['reference_id'].nunique().reindex(ROW_ORDER)

def canonical_variant_count(order_type):
    """N of SDOV/SIOV canonical variants paired against this construction's references --
    matches the paper's own Table 2/3 methodology, not a simple type count."""
    ref_ids = df[(df['is_original'] == 1) & (df['order_type'] == order_type)]['reference_id'].unique()
    sub = df[df['reference_id'].isin(ref_ids)]
    return len(sub[(sub['is_original'] == 0) & (sub['order_type'] == 'Canonical')])

def canonical_ref_variant_count():
    """N of non-canonical (fronted-type) variants paired against Canonical references --
    the reverse-direction comparison."""
    ref_ids = df[(df['is_original'] == 1) & (df['order_type'] == 'Canonical')]['reference_id'].unique()
    sub = df[df['reference_id'].isin(ref_ids)]
    return len(sub[(sub['is_original'] == 0) & (sub['order_type'] != 'Canonical')])

variants = pd.Series({
    'DO-fronted': canonical_variant_count('DO-fronted'),
    'IO-fronted': canonical_variant_count('IO-fronted'),
    'Canonical': canonical_ref_variant_count(),
}).reindex(ROW_ORDER)

def fmt_count_pct(counts):
    total = counts.sum()
    return counts.apply(lambda c: f'{c:,} ({c / total * 100:.2f}%)')

table1b = pd.DataFrame({
    'Pattern': [PATTERNS[o] for o in ROW_ORDER],
    'Reference': fmt_count_pct(refs).values,
    'Variant': fmt_count_pct(variants).values,
}, index=pd.Index(ROW_ORDER, name='Dataset ordering type'))
n_vars_total_1b = int((df['is_original'] == 0).sum())
table1b.loc['Full dataset'] = ['', f'{refs.sum():,}', f'{n_vars_total_1b:,}']
table1b

### Table 1c -- the published paper's own dataset composition (for comparison)

This is Ranjan & van Schijndel (2024)'s **own published** Table 1, reproduced here verbatim for
side-by-side comparison with our live Tables 1a/1b -- it is **hardcoded**, the second (and last)
explicitly-flagged hardcoded exception in this notebook alongside `PAPER_ACC` in Part 4, since it
is an externally published number, not something this pipeline computes.

The paper's totals (1,996 reference sentences / 72,833 variants, including an `O-fronted`
category) differ from our live totals above (2,320 references / 78,865 variants, no `O-fronted`
category) -- a known, already-documented difference in filtering criteria and a later retirement
of the `O-fronted` label from this project's own `classify_order()` (see the README's "A note on
data history" section), not a bug in either number.

In [11]:
table1c = pd.DataFrame([
    {'Dataset ordering type': 'O-fronted',    'Pattern': 'Ref: OSV; Var: SOV',   'Reference': '233 (11.67%)',   'Variant': '2,834 (3.89%)'},
    {'Dataset ordering type': 'DO-fronted',   'Pattern': 'Ref: DOSV; Var: SDOV', 'Reference': '133 (6.67%)',    'Variant': '1,663 (2.28%)'},
    {'Dataset ordering type': 'IO-fronted',   'Pattern': 'Ref: IOSV; Var: SIOV', 'Reference': '101 (5.06%)',    'Variant': '1,353 (1.86%)'},
    {'Dataset ordering type': 'Canonical',    'Pattern': 'Ref: SOV; Var: OSV',   'Reference': '1,763 (88.33%)', 'Variant': '30,851 (42.36%)'},
    {'Dataset ordering type': 'Full dataset', 'Pattern': '',                    'Reference': '1,996',           'Variant': '72,833'},
]).set_index('Dataset ordering type')
table1c

,Pattern,Reference,Variant
Dataset ordering type,,,
O-fronted,Ref: OSV; Var: SOV,233 (11.67%),"2,834 (3.89%)"
DO-fronted,Ref: DOSV; Var: SDOV,133 (6.67%),"1,663 (2.28%)"
IO-fronted,Ref: IOSV; Var: SIOV,101 (5.06%),"1,353 (1.86%)"
Canonical,Ref: SOV; Var: OSV,"1,763 (88.33%)","30,851 (42.36%)"
Full dataset,,"1,996","72,833"


## Part 3 [MTP-1] -- The pairwise classifier (shared by every experiment below)

Same method the paper uses: for each reference sentence, take the signed feature difference between the attested order and each generated canonical-order variant, fit a logistic regression on the pooled differences, evaluate with 10-fold stratified CV.

In [12]:
def build_pairs(df, feat_cols, order_type):
    ref_ids = df[(df['is_original'] == 1) & (df['order_type'] == order_type)]['reference_id'].unique()
    sub = df[df['reference_id'].isin(ref_ids)]
    Xs, ys = [], []
    for _, grp in sub.groupby('reference_id'):
        rf = grp[grp['is_original'] == 1]
        vr = grp[(grp['is_original'] == 0) & (grp['order_type'] == 'Canonical')]
        if len(rf) == 0 or len(vr) == 0:
            continue
        rfv = rf.iloc[0][feat_cols].values.astype(float)
        if np.any(np.isnan(rfv)):
            continue
        for _, v in vr.iterrows():
            vfv = v[feat_cols].values.astype(float)
            if np.any(np.isnan(vfv)):
                continue
            delta = rfv - vfv
            Xs.append(delta); ys.append(1)
            Xs.append(-delta); ys.append(0)
    if not Xs:
        return None, None
    return np.array(Xs), np.array(ys)

def run_cv(X, y, n_folds=10, seed=42):
    if X is None or len(X) == 0:
        return float('nan')
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    accs = []
    for tr, te in cv.split(X, y):
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr]); Xte = sc.transform(X[te])
        lr = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=seed)
        lr.fit(Xtr, y[tr])
        accs.append(accuracy_score(y[te], lr.predict(Xte)))
    return np.mean(accs) * 100

print('Classifier helpers ready (identical methodology to scripts/run_llm_comparison.py).')

Classifier helpers ready (identical methodology to scripts/run_llm_comparison.py).


## Part 4 [MTP-1] -- Replication check: Table 2

Our 5-feature (LSTM) classifier, run live here, vs. the numbers Ranjan & van Schijndel (2024) report in their paper. `PAPER_ACC` below is the **only hardcoded block in this entire notebook** -- it's an externally *published* number we're comparing against, not something this pipeline produces.

**This is the end of MTP-1.** Everything from Part 5 onward is MTP-2.

In [13]:
LSTM_FEATS = ['dl', 'is_score', 'surprisal_pcfg', 'surprisal_trigram', 'surprisal_lstm']

X_do, y_do = build_pairs(df, LSTM_FEATS, 'DO-fronted')
X_io, y_io = build_pairs(df, LSTM_FEATS, 'IO-fronted')
acc_do = run_cv(X_do, y_do)
acc_io = run_cv(X_io, y_io)

PAPER_ACC = {'DO-fronted': 80.46, 'IO-fronted': 90.02}  # Ranjan & van Schijndel (2024), published -- the one hardcoded exception

table2 = pd.DataFrame({
    'Construction': ['DO-fronted', 'IO-fronted'],
    'Our accuracy, % (live)':        [round(acc_do, 2), round(acc_io, 2)],
    'Paper accuracy, % (published)': [PAPER_ACC['DO-fronted'], PAPER_ACC['IO-fronted']],
})
table2['Diff (pts)'] = (table2['Our accuracy, % (live)'] - table2['Paper accuracy, % (published)']).round(2)
table2

,Construction,"Our accuracy, % (live)","Paper accuracy, % (published)",Diff (pts)
0,DO-fronted,77.51,80.46,-2.95
1,IO-fronted,92.22,90.02,2.20


## Part 5 [MTP-2] -- LLM extension + 3 new syntactic features: Table 3 and Figure 1

Single-feature ablations, the full 5-feature model with the surprisal source swapped (LSTM / XGLM-1.7B / Gemma-3), the 3 new tool-based syntactic features (PCFG / Modelblocks / Dependency-parser surprisal) alone and combined, and the "everything combined" model. Identical experiment matrix to `scripts/run_llm_comparison.py` -- reproduced live here instead of loaded from its output CSV.

In [14]:
xglm_col, gemma_col = 'surprisal_xglm', 'surprisal_gemma'
BASE_FEATS   = ['dl', 'is_score', 'surprisal_pcfg', 'surprisal_trigram']
LSTM_FEATS   = BASE_FEATS + ['surprisal_lstm']
XGLM_FEATS   = BASE_FEATS + [xglm_col]
GEMMA_FEATS  = BASE_FEATS + [gemma_col]
NEW3_FEATS   = ['surprisal_pcfg', 'surprisal_modelblocks', 'surprisal_depparse']
NON_LLM_ALL7 = LSTM_FEATS + ['surprisal_modelblocks', 'surprisal_depparse']
EVERYTHING   = BASE_FEATS + ['surprisal_lstm', xglm_col, gemma_col, 'surprisal_modelblocks', 'surprisal_depparse']

experiments = [
    ('Trigram only',                 ['surprisal_trigram']),
    ('DL only',                      ['dl']),
    ('IS score only',                ['is_score']),
    ('LSTM only',           ['surprisal_lstm']),
    ('XGLM only',                    [xglm_col]),
    ('Gemma-3 only',                 [gemma_col]),
    ('PCFG (tool) only',             ['surprisal_pcfg']),
    ('Modelblocks only',             ['surprisal_modelblocks']),
    ('Dep-parser only',              ['surprisal_depparse']),
    ('All 5 (original paper)',       LSTM_FEATS),
    ('LSTM -> XGLM replacement',     XGLM_FEATS),
    ('LSTM -> Gemma-3 replacement',  GEMMA_FEATS),
    ('3 new tool features combined', NEW3_FEATS),
    ('All 7 non-LLM (no XGLM/Gemma)', NON_LLM_ALL7),
    ('Everything combined',          EVERYTHING),
]

rows = []
for label, feats in experiments:
    row = {'Experiment': label}
    for ot in ['DO-fronted', 'IO-fronted']:
        X, y = build_pairs(df, feats, ot)
        row[ot] = round(run_cv(X, y), 1)
    rows.append(row)
    print(f'  {label:<32} DO={row["DO-fronted"]:.1f}%  IO={row["IO-fronted"]:.1f}%')

table3 = pd.DataFrame(rows)
table3

  Trigram only                     DO=76.0%  IO=88.7%


  DL only                          DO=61.4%  IO=65.3%


  IS score only                    DO=51.5%  IO=52.0%


  LSTM only                        DO=74.8%  IO=88.5%


  XGLM only                        DO=88.2%  IO=92.3%


  Gemma-3 only                     DO=78.0%  IO=78.7%


... (6 outputs left out)

  3 new tool features combined     DO=53.8%  IO=52.8%


  All 7 non-LLM (no XGLM/Gemma)    DO=80.0%  IO=93.1%


  Everything combined              DO=89.5%  IO=95.1%


,Experiment,DO-fronted,IO-fronted
0,Trigram only,76.0,88.7
1,DL only,61.4,65.3
2,IS score only,51.5,52.0
3,LSTM only,74.8,88.5
4,XGLM only,88.2,92.3
5,Gemma-3 only,78.0,78.7
6,PCFG (tool) only,53.3,49.3
7,Modelblocks only,52.1,50.0
8,Dep-parser only,53.1,53.2
9,All 5 (original paper),77.5,92.2


In [15]:
# Figure 1 -- full 5-feature model accuracy by construction, 3 surprisal sources
t3 = table3.set_index('Experiment')
full_rows = {
    "LSTM\n(paper's setup)": 'All 5 (original paper)',
    'LSTM -> XGLM-1.7B':                'LSTM -> XGLM replacement',
    'LSTM -> Gemma-3':                  'LSTM -> Gemma-3 replacement',
}

fig, ax = plt.subplots(figsize=(6, 3.4))
x = np.arange(2)
width = 0.26
colors = [C_LSTM, C_XGLM, C_GEMMA]
for i, (model_label, exp_label) in enumerate(full_rows.items()):
    vals = [t3.loc[exp_label, 'DO-fronted'], t3.loc[exp_label, 'IO-fronted']]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=model_label, color=colors[i], zorder=3)
    bar_labels(ax, bars)
ax.set_xticks(x); ax.set_xticklabels(['DO-fronted', 'IO-fronted'])
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0, 100)
ax.set_title('Why LLMs beat the LSTM (live)')
style_axes(ax)
ax.legend(fontsize=8, frameon=False, loc='lower right')
plt.tight_layout()
fig1_path = os.path.join(FIGURES_DIR, 'fig1_llm_vs_lstm.png')
fig.savefig(fig1_path, dpi=150, bbox_inches='tight')
print(f'Saved -> {fig1_path}')
plt.show()

Saved -> E:\hindi_wordorder_replication\results\figures\fig1_llm_vs_lstm.png


## Part 6 [MTP-2] -- Coefficient significance: Table 4

Fits the full GLM (statsmodels, so we get standard errors / Z / VIF, not just CV accuracy) for each surprisal source / feature set, for both constructions.

In [16]:
def glm_coefs(df, feat_cols, feat_labels, order_type):
    ref_ids = df[(df['is_original'] == 1) & (df['order_type'] == order_type)]['reference_id'].unique()
    sub = df[df['reference_id'].isin(ref_ids)]
    Xs, ys = [], []
    for _, grp in sub.groupby('reference_id'):
        rf = grp[grp['is_original'] == 1]
        vr = grp[(grp['is_original'] == 0) & (grp['order_type'] == 'Canonical')]
        if len(rf) == 0 or len(vr) == 0:
            continue
        rfv = rf.iloc[0][feat_cols].values.astype(float)
        for _, v in vr.iterrows():
            vfv = v[feat_cols].values.astype(float)
            if np.any(np.isnan(rfv)) or np.any(np.isnan(vfv)):
                continue
            Xs.append(rfv - vfv); ys.append(1)
            Xs.append(vfv - rfv); ys.append(0)
    X_raw, y = np.array(Xs), np.array(ys)
    X_sc = StandardScaler().fit_transform(X_raw)
    X_c = sm.add_constant(X_sc)
    res = sm.Logit(y, X_c).fit(disp=0, maxiter=300)
    vifs = [variance_inflation_factor(X_c, i) for i in range(X_c.shape[1])]
    labels = ['Intercept'] + feat_labels
    return pd.DataFrame({'Predictor': labels, 'Estimate': res.params, 'Std_Error': res.bse,
                         'Z_value': res.tvalues, 'P_value': res.pvalues, 'VIF': vifs})

BASE_LABELS = ['Dependency length', 'IS score', 'PCFG surprisal', 'Trigram surprisal']
model_defs = {
    'All-5 original (LSTM)':      (LSTM_FEATS,  BASE_LABELS + ['LSTM surprisal']),
    'LSTM -> XGLM replacement':   (XGLM_FEATS,  BASE_LABELS + ['XGLM surprisal']),
    'LSTM -> Gemma-3 replacement':(GEMMA_FEATS, BASE_LABELS + ['Gemma-3 surprisal']),
    'All 7 non-LLM':              (NON_LLM_ALL7, BASE_LABELS + ['LSTM surprisal', 'Modelblocks surprisal', 'Dep-parser surprisal']),
    'Everything combined':        (EVERYTHING,  BASE_LABELS + ['LSTM surprisal', 'XGLM surprisal', 'Gemma-3 surprisal', 'Modelblocks surprisal', 'Dep-parser surprisal']),
}

coef_rows = []
for construction in ['DO-fronted', 'IO-fronted']:
    for model_name, (feats, labels) in model_defs.items():
        res = glm_coefs(df, feats, labels, construction)
        for _, r in res.iloc[1:].iterrows():  # skip intercept
            p = r['P_value']
            coef_rows.append({
                'Construction': construction, 'Model': model_name, 'Predictor': r['Predictor'],
                'Estimate': round(r['Estimate'], 4), 'P_value': round(p, 6),
                'Sig': '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '',
            })
    print(f'  Fitted all 5 GLMs for {construction}')

table4 = pd.DataFrame(coef_rows)
table4

  Fitted all 5 GLMs for DO-fronted


  Fitted all 5 GLMs for IO-fronted


,Construction,Model,Predictor,Estimate,P_value,Sig
0,DO-fronted,All-5 original (LSTM),Dependency length,-0.5414,0.000000,***
1,DO-fronted,All-5 original (LSTM),IS score,-0.2083,0.000028,***
2,DO-fronted,All-5 original (LSTM),PCFG surprisal,0.1599,0.000981,***
3,DO-fronted,All-5 original (LSTM),Trigram surprisal,-1.4426,0.000000,***
4,DO-fronted,All-5 original (LSTM),LSTM surprisal,-0.7891,0.000000,***
...,...,...,...,...,...,...
57,IO-fronted,Everything combined,LSTM surprisal,-0.2935,0.456807,
58,IO-fronted,Everything combined,XGLM surprisal,-3.7085,0.000000,***
59,IO-fronted,Everything combined,Gemma-3 surprisal,-1.4331,0.000000,***
60,IO-fronted,Everything combined,Modelblocks surprisal,-0.1223,0.411658,


### Table 4b -- Everything-combined coefficients, by construction (presentation view)

Pivots `table4` above (computed live, no new fitting) into a compact Feature x Construction view for the "Everything combined" model -- the one used in Figure 3.

In [ ]:
feature_order = ['Dependency length', 'IS score', 'PCFG surprisal', 'Trigram surprisal',
                 'LSTM surprisal', 'XGLM surprisal', 'Gemma-3 surprisal',
                 'Modelblocks surprisal', 'Dep-parser surprisal']
feature_display = {
    'Dependency length': 'Dependency length', 'IS score': 'Information status',
    'PCFG surprisal': 'PCFG surprisal', 'Trigram surprisal': 'Trigram surprisal',
    'LSTM surprisal': 'Adaptive LSTM surprisal', 'XGLM surprisal': 'XGLM surprisal',
    'Gemma-3 surprisal': 'Gemma-3 surprisal', 'Modelblocks surprisal': 'Modelblocks surprisal',
    'Dep-parser surprisal': 'Dependency-parser surprisal',
}

ec = table4[table4['Model'] == 'Everything combined']
do_ec = ec[ec['Construction'] == 'DO-fronted'].set_index('Predictor')
io_ec = ec[ec['Construction'] == 'IO-fronted'].set_index('Predictor')

table4b = pd.DataFrame({
    'Feature':           [feature_display[f] for f in feature_order],
    'DO-fronted (β)':    [do_ec.loc[f, 'Estimate'] for f in feature_order],
    'IO-fronted (β)':    [io_ec.loc[f, 'Estimate'] for f in feature_order],
    'DO significance':   [do_ec.loc[f, 'Sig'] or 'n.s.' for f in feature_order],
    'IO significance':   [io_ec.loc[f, 'Sig'] or 'n.s.' for f in feature_order],
}).set_index('Feature')
table4b

## Part 7 [MTP-2] -- Why do LLMs win? Table 5 and Figure 2

Per-sentence: does the LSTM surprisal correctly rank the attested order above the mean of its canonical-order variants, and does XGLM? Stratified by dependency length (sentence complexity) and by out-of-vocabulary rate against the LSTM's own closed vocabulary.

In [17]:
with open(VOCAB_JSON, encoding='utf-8') as f:
    vocab = json.load(f)
w2i = vocab.get('w2i', vocab)  # handle either {'w2i': {...}} or a flat dict
print(f'LSTM vocabulary size: {len(w2i):,} types')

records = []
for order_type in ['DO-fronted', 'IO-fronted']:
    corpus_rows = df[(df['is_original'] == 1) & (df['order_type'] == order_type)]
    for _, corp in corpus_rows.iterrows():
        cands = df[(df['reference_id'] == corp['reference_id']) &
                   (df['is_original'] == 0) & (df['order_type'] == 'Canonical')]
        if len(cands) == 0:
            continue
        lstm_correct = corp['surprisal_lstm'] < cands['surprisal_lstm'].mean()
        xglm_correct = corp['surprisal_xglm'] < cands['surprisal_xglm'].mean()
        words = str(corp['text']).split()
        oov = sum(1 for w in words if w not in w2i) / len(words) if words else np.nan
        records.append({'reference_id': corp['reference_id'], 'order_type': order_type,
                        'text': corp['text'], 'dl': corp['dl'], 'oov_rate': oov,
                        'lstm_correct': lstm_correct, 'xglm_correct': xglm_correct})

err_df = pd.DataFrame(records)
print(f'Per-sentence comparisons: {len(err_df)}')

def acc_row(mask, label):
    sub = err_df[mask]
    return {'Subset': label, 'n': len(sub),
            'LSTM acc. (%)': round(100 * sub['lstm_correct'].mean(), 1),
            'XGLM acc. (%)': round(100 * sub['xglm_correct'].mean(), 1)}

table5 = pd.DataFrame([
    acc_row(err_df['dl'] > 50, 'Dependency length > 50'),
    acc_row(err_df['dl'] <= 50, 'Dependency length <= 50'),
    acc_row(err_df['oov_rate'] > 0.1, 'OOV rate > 10%'),
    acc_row(err_df['oov_rate'] <= 0.1, 'OOV rate <= 10%'),
    acc_row(pd.Series(True, index=err_df.index), 'Overall'),
])
table5

LSTM vocabulary size: 112,876 types


Per-sentence comparisons: 247


,Subset,n,LSTM acc. (%),XGLM acc. (%)
0,Dependency length > 50,84,75.0,91.7
1,Dependency length <= 50,163,73.0,81.6
2,OOV rate > 10%,121,70.2,81.8
3,OOV rate <= 10%,126,77.0,88.1
4,Overall,247,73.7,85.0


In [18]:
# Figure 2 -- stratified accuracy by dependency length (a) and OOV rate (b)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(7.5, 3.4))
width = 0.32
x = np.arange(2)
t5 = table5.set_index('Subset')

b1 = axL.bar(x - width/2, [t5.loc['Dependency length > 50', 'LSTM acc. (%)'], t5.loc['Dependency length <= 50', 'LSTM acc. (%)']],
             width, label='LSTM', color=C_LSTM, zorder=3)
b2 = axL.bar(x + width/2, [t5.loc['Dependency length > 50', 'XGLM acc. (%)'], t5.loc['Dependency length <= 50', 'XGLM acc. (%)']],
             width, label='XGLM-1.7B', color=C_XGLM, zorder=3)
axL.set_xticks(x); axL.set_xticklabels(['DL > 50', 'DL <= 50'])
axL.set_ylabel('Accuracy (%)'); axL.set_ylim(0, 100); axL.set_title('By dependency length')
style_axes(axL); bar_labels(axL, b1); bar_labels(axL, b2)

b3 = axR.bar(x - width/2, [t5.loc['OOV rate > 10%', 'LSTM acc. (%)'], t5.loc['OOV rate <= 10%', 'LSTM acc. (%)']],
             width, label='LSTM', color=C_LSTM, zorder=3)
b4 = axR.bar(x + width/2, [t5.loc['OOV rate > 10%', 'XGLM acc. (%)'], t5.loc['OOV rate <= 10%', 'XGLM acc. (%)']],
             width, label='XGLM-1.7B', color=C_XGLM, zorder=3)
axR.set_xticks(x); axR.set_xticklabels(['OOV > 10%', 'OOV <= 10%'])
axR.set_ylim(0, 100); axR.set_title('By out-of-vocabulary rate')
style_axes(axR); bar_labels(axR, b3); bar_labels(axR, b4)
axR.legend(fontsize=8, frameon=False, loc='lower right')

plt.tight_layout()
plt.show()

### Sentence-level error analysis -- where does XGLM beat the LSTM?

Same per-sentence correctness check used for Table 5 above, split into the 2x2: both correct, only XGLM correct (the LLM wins), only LSTM correct (the LLM loses), and both wrong -- with a few actual example sentences pulled live from each bucket.

In [ ]:
both_correct = err_df[err_df['lstm_correct'] & err_df['xglm_correct']]
llm_wins     = err_df[~err_df['lstm_correct'] & err_df['xglm_correct']]
llm_loses    = err_df[err_df['lstm_correct'] & ~err_df['xglm_correct']]
both_wrong   = err_df[~err_df['lstm_correct'] & ~err_df['xglm_correct']]

n = len(err_df)
print(f'Sentence-level error breakdown (n={n:,} corpus sentences vs. their canonical variants):')
print(f'  Both correct             : {len(both_correct):>4}  ({100*len(both_correct)/n:5.1f}%)')
print(f'  XGLM right, LSTM wrong   : {len(llm_wins):>4}  ({100*len(llm_wins)/n:5.1f}%)   <- LLM wins')
print(f'  LSTM right, XGLM wrong   : {len(llm_loses):>4}  ({100*len(llm_loses)/n:5.1f}%)   <- LLM loses')
print(f'  Both wrong               : {len(both_wrong):>4}  ({100*len(both_wrong)/n:5.1f}%)')

print('\nExample sentences where XGLM succeeds but the LSTM fails (sorted by dependency length):')
for _, r in llm_wins.sort_values('dl', ascending=False).head(5).iterrows():
    print(f"  [dl={r['dl']:.0f}, oov={r['oov_rate']*100:.0f}%] {r['text']}")

print('\nExample sentences where the LSTM succeeds but XGLM fails:')
for _, r in llm_loses.sort_values('dl').head(5).iterrows():
    print(f"  [dl={r['dl']:.0f}, oov={r['oov_rate']*100:.0f}%] {r['text']}")

error_summary = pd.DataFrame([
    {'Outcome': 'Both correct',           'n': len(both_correct), '%': round(100*len(both_correct)/n, 1)},
    {'Outcome': 'XGLM right, LSTM wrong', 'n': len(llm_wins),     '%': round(100*len(llm_wins)/n, 1)},
    {'Outcome': 'LSTM right, XGLM wrong', 'n': len(llm_loses),    '%': round(100*len(llm_loses)/n, 1)},
    {'Outcome': 'Both wrong',             'n': len(both_wrong),   '%': round(100*len(both_wrong)/n, 1)},
])
error_summary

## Part 8 -- Putting it all together

In [22]:
t3 = table3.set_index('Experiment')
avg_all5      = (t3.loc['All 5 (original paper)', 'DO-fronted']      + t3.loc['All 5 (original paper)', 'IO-fronted'])      / 2
avg_everything = (t3.loc['Everything combined', 'DO-fronted']        + t3.loc['Everything combined', 'IO-fronted'])         / 2

summary = pd.DataFrame([
    {'Pipeline stage': 'MTP-1: All-5 original (LSTM)',             'DO-fronted %': t3.loc['All 5 (original paper)', 'DO-fronted'], 'IO-fronted %': t3.loc['All 5 (original paper)', 'IO-fronted']},
    {'Pipeline stage': 'MTP-2: LSTM -> XGLM-1.7B',                  'DO-fronted %': t3.loc['LSTM -> XGLM replacement', 'DO-fronted'], 'IO-fronted %': t3.loc['LSTM -> XGLM replacement', 'IO-fronted']},
    {'Pipeline stage': 'MTP-2: Everything combined (+3 new feats)', 'DO-fronted %': t3.loc['Everything combined', 'DO-fronted'], 'IO-fronted %': t3.loc['Everything combined', 'IO-fronted']},
])
summary

In [23]:
# Figure 3 -- final synthesis
fig, ax = plt.subplots(figsize=(6, 3.6))
labels = ['MTP-1\nAll-5 (LSTM)', 'MTP-2\nLSTM->XGLM', 'MTP-2\nEverything\ncombined']
values = [avg_all5, t3.loc['LSTM -> XGLM replacement', ['DO-fronted', 'IO-fronted']].mean(),
          avg_everything]
bars = ax.bar(range(len(labels)), values, color=[C_LSTM, C_XGLM, C_NEW], zorder=3, width=0.6)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Accuracy (%, avg DO/IO where applicable)'); ax.set_ylim(0, 100)
ax.set_title('MTP-1 -> MTP-2: every stage, computed live in this notebook')
style_axes(ax); bar_labels(ax, bars)
plt.tight_layout()
plt.show()

print('\nAll numbers above were computed in this run -- nothing was pasted in.')